# Smart Pet Vision — ONNX numerical parity repair (Phase 2 only)
**READ FIRST:** This is a deterministic verification notebook, **not** a report of successful parity. Use only an authorized private Drive working copy. It preserves Phase 1 and the 30 Oct model files. The project raw-head threshold stays **0.001**. Do not deploy/export over existing files. A successful ONNX parity still does not authorize publication.

The runner compares the frozen model and a new ONNX export (if requested) on **four seeded random tensors plus 50 preselected, previously inspected team photos**. Backend is **CPU for numerical parity** even on a T4 Colab. The T4 latency benchmark is a separate milestone.

Prerequisites: copy the linked Stage2.1 source ZIP, original `best.pt`, original `best.onnx`, and the team's authorized photos folder into *your* Drive, or adjust the paths in the next cell. NEVER add private images or API credentials to GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
# Set these to existing resources in your private My Drive (a shortcut to shared capstone folder is OK).
BASE = Path('/content/drive/MyDrive/SPV_Phase2_ONNX_Work')
SOURCE_ZIP = BASE / 'SmartPetVision_Phase2_ONNX_Repair_Gate_v0.3_2026-10-31.zip'
PT_WEIGHTS = BASE / 'best.pt'
ONNX_BASELINE = BASE / 'best.onnx'
TEAM_IMAGES = BASE / 'team_images'  # 401 authorized images, local-only parity usage
for item in [SOURCE_ZIP,PT_WEIGHTS,ONNX_BASELINE,TEAM_IMAGES]:
    if not item.exists(): raise FileNotFoundError(f'Please provide authorized Drive asset: {item}')
print('Prerequisite Drive assets found.')

In [ ]:
# Dedicated environment only. Keep the Phase 1/Phase 2 training directories untouched.
%pip -q install "ultralytics==8.4.182" "onnxruntime==1.31.0" "onnx==1.23.2" "pillow" "numpy" "torch" "torchvision" "pandas" "fastapi" "httpx" "uvicorn" "pyyaml"
import importlib.metadata
for package in ('torch','ultralytics','onnxruntime','onnx'):
    print(package,importlib.metadata.version(package))

In [ ]:
import zipfile, shutil, json, hashlib
WORK=Path('/content/SPV_phase2_onnx_stage2_1')
if WORK.exists(): raise FileExistsError(f'Refusing to overwrite an old validation workspace: {WORK}')
WORK.mkdir()
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    for member in archive.namelist():
        dest=(WORK/member).resolve()
        if WORK.resolve() not in dest.parents and dest!=WORK.resolve(): raise ValueError('Unsafe archive path')
    archive.extractall(WORK)
print('Extracted to',WORK)
# Find exactly one approved source tree with 'tools/run_onnx_repair.py'
options=list(WORK.rglob('tools/run_onnx_repair.py'))
if len(options)!=1:raise RuntimeError(f'Found {len(options)} source trees; expected exactly one')
REPO=options[0].parents[1]
print('Approved candidate source:',REPO)

In [ ]:
# Freeze a fixed parity cohort. These are NOT unseen independent model-quality test photos.
import sys
sys.path.insert(0,str(REPO/'tools'))
from onnx_parity_diagnostics import sha256_file
selected = '0003 0034 0038 0040 0041 0065 0066 0069 0071 0083 0084 0089 0090 0093 0094 0104 0116 0118 0133 0148 0168 0169 0175 0192 0196 0201 0211 0217 0222 0230 0243 0244 0247 0248 0252 0259 0264 0265 0290 0291 0296 0325 0330 0335 0339 0364 0381 0382 0398 0412'.split()
rows=[]
for id in selected:
    item=TEAM_IMAGES/f'team_{id}.jpg'
    if not item.is_file(): raise FileNotFoundError(item)
    rows.append({'file':item.name,'sha256':sha256_file(item)})
COHORT=WORK/'frozen_parity_cohort_50_PRIVATE.json'
with COHORT.open('x') as fp:json.dump(rows,fp,indent=2)
print('Private parity cohort',len(rows), 'manifest hash',sha256_file(COHORT))

In [ ]:
import subprocess,sys,json
REPORT=WORK/'results_baseline'
cmd=[sys.executable,str(REPO/'tools/run_onnx_repair.py'),
    '--repo',str(REPO),'--weights',str(PT_WEIGHTS),'--baseline-onnx',str(ONNX_BASELINE),
    '--image-root',str(TEAM_IMAGES),'--cohort',str(COHORT),
    '--baseline',str(REPO/'verification/stage2/APPROVED_BASELINE_MANIFEST_2026-10-30.json'),
    '--output',str(REPORT),'--opt-levels','DISABLE_ALL','ENABLE_ALL']
process=subprocess.run(cmd,capture_output=True,text=True)
print(process.stdout)
if process.stderr:print(process.stderr[-2000:])
result=json.loads((REPORT/'onnx_repair_matrix.json').read_text())
print('BASELINE STATUS:',result['status'])
print('Eligible variants:',result.get('eligible_variants',[]))
print('All model variants:',[(x['name'],x.get('decision',{}).get('status')) for x in result['variants']])
# Failing is a valid scientific result. Do not change the acceptance tolerance.

In [ ]:
# Optional: evaluate a *new* export, never overwriting the pre-existing best.onnx.
# This exports to a unique new file from a temporary copy of the frozen best.pt.
# Run only after reviewing the baseline report; compare both baseline and candidate.
RUN_CANDIDATE = False
if RUN_CANDIDATE:
    candidate=WORK/'exports/detector_candidate_opset17.onnx'
    out=WORK/'results_reexport_candidate'
    cmd2=cmd.copy()
    cmd2[cmd2.index(str(REPORT))]=str(out)
    cmd2+=['--candidate-output',str(candidate)]
    p=subprocess.run(cmd2,capture_output=True,text=True)
    print(p.stdout,p.stderr[-2000:])
    print(json.loads((out/'onnx_repair_matrix.json').read_text())['status'])

## Interpretation and next step
- **PASS:** independent specialist review is required, including full code-source matching, valid result population, and T4 benchmark before any release. Do not push automatically.
- **FAIL:** preserve every test; keep the supervised **PyTorch** inference backend. The project-specific 0.001 raw-head criterion remains unpassed; do not re-label final-box agreement a pass.
- **BLOCKED:** do not infer a pass, resolve the exact stated dependency or provenance issue, then rerun to a *new* result directory.
- Even a passing parity gate does **not** resolve the known multiple-animal false-ready problem. Every technical Ready output still requires staff approval.
- Copy only JSON logs and source checks to Drive when reviewing. Do not upload team photos or secrets to a public GitHub repository.